# Qdrant Demo

This notebook demonstrates how we will use Qdrant to store and retrieve embeddings. We begin by importing the neccesary libraires

In [8]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance
from ollama import Client as OllamaClient, ChatResponse
import requests
from typing import Mapping, Any, Optional, Sequence
import os

Next we set the url of our locally deployed Ollama instance

In [9]:
url = "http://localhost:11434/api/embed"

Next we define a function to get an Ollama client

In [10]:
_client: Optional["OllamaClient"] = None

def _get_client() -> OllamaClient:
    global _client
    if _client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _client = OllamaClient(host=host)
    return _client

This function accepts a strings and returns the corresponding embedding (vector) from nomic-text-embed running in Ollama

In [11]:
def embed(text:str) -> Sequence[float]:
    _client = _get_client()
    response: ChatResponse = _client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] # type: ignore

Now we initialize a Qdrant client. NOTE: Long term, adjust this to match the "get client" paradigm

In [16]:
qdrant_client = QdrantClient(host="localhost", port=6333) # mirror Ollama setup

Create a collection of embeddings of appropriatte size in qdrant

In [17]:
qdrant_client.recreate_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

/var/folders/nb/0qk42j7j0bq93_44dknp53s00000gn/T/ipykernel_7646/3554724465.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True